# Model Hosting on AWS

In this notebook, we'll deploy our optimized models to AWS SageMaker for inference. We'll compare the performance and cost of hosting the baseline models versus the optimized models.

## 1. Import Dependencies

In [ ]:
import os
import json
import time
import boto3
import sagemaker
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sagemaker.huggingface import HuggingFaceModel
from datetime import datetime

# Import workshop configuration
from workshop_config import S3_BUCKET, AWS_REGION, SAGEMAKER_ROLE_ARN

## 2. Load Model Information

In [ ]:
# Load model information
with open('model_info.json', 'r') as f:
    model_info = json.load(f)

# Load baseline metrics
with open('baseline_metrics.json', 'r') as f:
    baseline_metrics = json.load(f)

# Load quantized metrics if available
try:
    with open('quantized_metrics.json', 'r') as f:
        quantized_metrics = json.load(f)
    print("Loaded quantized metrics")
except FileNotFoundError:
    quantized_metrics = {}
    print("No quantized metrics found")

# Load pruned metrics if available
try:
    with open('pruned_metrics.json', 'r') as f:
        pruned_metrics = json.load(f)
    print("Loaded pruned metrics")
except FileNotFoundError:
    pruned_metrics = {}
    print("No pruned metrics found")

## 3. Define Sample Inputs for Each Task

In [ ]:
# Define sample inputs for each task
sample_inputs = {
    "sentiment_analysis": {
        "inputs": "I really enjoyed this movie. The acting was superb and the plot was engaging."
    },
    "ner": {
        "inputs": "Jeff Bezos founded Amazon in 1994 and the company is headquartered in Seattle, Washington."
    },
    "question_answering": {
        "inputs": {
            "question": "What is machine learning?",
            "context": "Machine learning is a branch of artificial intelligence that focuses on building systems that learn from data."
        }
    },
    "masked_lm": {
        "inputs": "The [MASK] is a large language model trained by OpenAI."
    }
}

## 4. Create SageMaker Session

In [ ]:
# Create SageMaker session
sagemaker_session = sagemaker.Session()
role = SAGEMAKER_ROLE_ARN

# Create S3 client
s3_client = boto3.client('s3')

print(f"SageMaker session created in region {AWS_REGION}")
print(f"Using role: {role}")

## 5. Create Function to Deploy Model to SageMaker

In [ ]:
def deploy_model_to_sagemaker(model_path, model_name, task, instance_type="ml.g4dn.xlarge"):
    """Deploy a model to SageMaker."""
    # Create a unique endpoint name
    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    endpoint_name = f"{model_name.replace('/', '-')}-{timestamp}"
    
    print(f"Deploying {model_name} to SageMaker...")
    
    # Create Hugging Face model
    huggingface_model = HuggingFaceModel(
        model_data=model_path,
        role=role,
        transformers_version="4.26.0",
        pytorch_version="1.13.1",
        py_version="py39",
        sagemaker_session=sagemaker_session
    )
    
    # Deploy model to endpoint
    predictor = huggingface_model.deploy(
        initial_instance_count=1,
        instance_type=instance_type,
        endpoint_name=endpoint_name
    )
    
    print(f"Model deployed to endpoint: {endpoint_name}")
    
    return {
        "endpoint_name": endpoint_name,
        "predictor": predictor
    }

## 6. Create Function to Package Model for SageMaker

In [ ]:
def package_model_for_sagemaker(model_path, s3_bucket, model_name):
    """Package a model for SageMaker deployment."""
    import tarfile
    import tempfile
    import shutil
    
    # Create a temporary directory
    temp_dir = tempfile.mkdtemp()
    model_dir = os.path.join(temp_dir, "model")
    os.makedirs(model_dir, exist_ok=True)
    
    # Copy model files to temporary directory
    for item in os.listdir(model_path):
        s = os.path.join(model_path, item)
        d = os.path.join(model_dir, item)
        if os.path.isdir(s):
            shutil.copytree(s, d)
        else:
            shutil.copy2(s, d)
    
    # Create tar.gz file
    tar_path = os.path.join(temp_dir, "model.tar.gz")
    with tarfile.open(tar_path, "w:gz") as tar:
        tar.add(model_dir, arcname="")
    
    # Upload to S3
    s3_key = f"models/{model_name.replace('/', '-')}/model.tar.gz"
    s3_client.upload_file(tar_path, s3_bucket, s3_key)
    
    # Clean up
    shutil.rmtree(temp_dir)
    
    return f"s3://{s3_bucket}/{s3_key}"

## 7. Select Models to Deploy

Let's select one model from each optimization technique to deploy.

In [ ]:
# Select a model to deploy (e.g., sentiment analysis)
model_key = "sentiment_analysis"

# Get model paths
baseline_path = model_info[model_key]["local_path"]
quantized_path = quantized_metrics.get(model_key, {}).get("quantized_path", None)
pruned_path = pruned_metrics.get(model_key, {}).get("pruned_path", None)

print(f"Selected model: {model_info[model_key]['model_name']}")
print(f"Baseline path: {baseline_path}")
print(f"Quantized path: {quantized_path}")
print(f"Pruned path: {pruned_path}")

## 8. Package Models for SageMaker

In [ ]:
# Package baseline model
baseline_s3_path = package_model_for_sagemaker(
    baseline_path, 
    S3_BUCKET, 
    f"{model_info[model_key]['model_name']}-baseline"
)
print(f"Baseline model packaged at {baseline_s3_path}")

# Package quantized model if available
quantized_s3_path = None
if quantized_path:
    quantized_s3_path = package_model_for_sagemaker(
        quantized_path, 
        S3_BUCKET, 
        f"{model_info[model_key]['model_name']}-quantized"
    )
    print(f"Quantized model packaged at {quantized_s3_path}")

# Package pruned model if available
pruned_s3_path = None
if pruned_path:
    pruned_s3_path = package_model_for_sagemaker(
        pruned_path, 
        S3_BUCKET, 
        f"{model_info[model_key]['model_name']}-pruned"
    )
    print(f"Pruned model packaged at {pruned_s3_path}")

## 9. Deploy Models to SageMaker

In [ ]:
# Deploy baseline model
baseline_deployment = deploy_model_to_sagemaker(
    baseline_s3_path,
    f"{model_info[model_key]['model_name']}-baseline",
    model_info[model_key]['task']
)

# Deploy quantized model if available
quantized_deployment = None
if quantized_s3_path:
    quantized_deployment = deploy_model_to_sagemaker(
        quantized_s3_path,
        f"{model_info[model_key]['model_name']}-quantized",
        model_info[model_key]['task']
    )

# Deploy pruned model if available
pruned_deployment = None
if pruned_s3_path:
    pruned_deployment = deploy_model_to_sagemaker(
        pruned_s3_path,
        f"{model_info[model_key]['model_name']}-pruned",
        model_info[model_key]['task']
    )

## 10. Test Inference on Deployed Models

In [ ]:
def test_inference(predictor, input_data):
    """Test inference on a deployed model."""
    # Measure inference time
    start_time = time.time()
    response = predictor.predict(input_data)
    end_time = time.time()
    
    inference_time = (end_time - start_time) * 1000  # Convert to ms
    
    return {
        "response": response,
        "inference_time": inference_time
    }

In [ ]:
# Get input data for the selected model
input_data = sample_inputs[model_key]

# Test baseline model
print("Testing baseline model...")
baseline_result = test_inference(baseline_deployment["predictor"], input_data)
print(f"Inference time: {baseline_result['inference_time']:.2f} ms")
print(f"Response: {baseline_result['response']}\n")

# Test quantized model if available
if quantized_deployment:
    print("Testing quantized model...")
    quantized_result = test_inference(quantized_deployment["predictor"], input_data)
    print(f"Inference time: {quantized_result['inference_time']:.2f} ms")
    print(f"Response: {quantized_result['response']}\n")

# Test pruned model if available
if pruned_deployment:
    print("Testing pruned model...")
    pruned_result = test_inference(pruned_deployment["predictor"], input_data)
    print(f"Inference time: {pruned_result['inference_time']:.2f} ms")
    print(f"Response: {pruned_result['response']}\n")

## 11. Compare Inference Performance

In [ ]:
# Collect inference times
inference_times = {
    "Baseline": baseline_result["inference_time"]
}

if quantized_deployment:
    inference_times["Quantized"] = quantized_result["inference_time"]

if pruned_deployment:
    inference_times["Pruned"] = pruned_result["inference_time"]

# Create DataFrame
inference_df = pd.DataFrame({
    "Model": list(inference_times.keys()),
    "Inference Time (ms)": list(inference_times.values())
})

# Plot inference times
plt.figure(figsize=(10, 6))
sns.barplot(x="Model", y="Inference Time (ms)", data=inference_df)
plt.title("Inference Time Comparison")
plt.ylabel("Inference Time (ms)")
plt.tight_layout()
plt.show()

## 12. Calculate Speedup and Cost Savings

In [ ]:
# Calculate speedup
speedup = {}
if quantized_deployment:
    speedup["Quantized"] = baseline_result["inference_time"] / quantized_result["inference_time"]

if pruned_deployment:
    speedup["Pruned"] = baseline_result["inference_time"] / pruned_result["inference_time"]

# Create DataFrame
speedup_df = pd.DataFrame({
    "Model": list(speedup.keys()),
    "Speedup Factor": list(speedup.values())
})

# Plot speedup
plt.figure(figsize=(10, 6))
sns.barplot(x="Model", y="Speedup Factor", data=speedup_df)
plt.title("Speedup Factor Compared to Baseline")
plt.ylabel("Speedup Factor (higher is better)")
plt.tight_layout()
plt.show()

In [ ]:
def estimate_monthly_cost(inference_time_ms, model_size_mb, requests_per_month=1000000):
    """Estimate monthly cost for running a model in production."""
    # Assumptions
    compute_cost_per_hour = 0.5  # $0.5 per hour for compute (e.g., ml.g4dn.xlarge)
    storage_cost_per_gb_month = 0.023  # $0.023 per GB-month for S3
    
    # Calculate compute cost
    inference_time_hours = (inference_time_ms * requests_per_month) / (1000 * 60 * 60)
    compute_cost = inference_time_hours * compute_cost_per_hour
    
    # Calculate storage cost
    storage_cost = (model_size_mb / 1024) * storage_cost_per_gb_month
    
    # Total cost
    total_cost = compute_cost + storage_cost
    
    return {
        "compute_cost": compute_cost,
        "storage_cost": storage_cost,
        "total_cost": total_cost
    }

In [ ]:
# Get model sizes
baseline_size = baseline_metrics[model_key]["model_size"]
quantized_size = quantized_metrics.get(model_key, {}).get("model_size", baseline_size)
pruned_size = pruned_metrics.get(model_key, {}).get("model_size", baseline_size)

# Estimate costs
baseline_cost = estimate_monthly_cost(baseline_result["inference_time"], baseline_size)
costs = {
    "Baseline": baseline_cost["total_cost"]
}

if quantized_deployment:
    quantized_cost = estimate_monthly_cost(quantized_result["inference_time"], quantized_size)
    costs["Quantized"] = quantized_cost["total_cost"]

if pruned_deployment:
    pruned_cost = estimate_monthly_cost(pruned_result["inference_time"], pruned_size)
    costs["Pruned"] = pruned_cost["total_cost"]

# Create DataFrame
cost_df = pd.DataFrame({
    "Model": list(costs.keys()),
    "Monthly Cost ($)": list(costs.values())
})

# Plot costs
plt.figure(figsize=(10, 6))
sns.barplot(x="Model", y="Monthly Cost ($)", data=cost_df)
plt.title("Estimated Monthly Cost (1M requests/month)")
plt.ylabel("Monthly Cost ($)")
plt.tight_layout()
plt.show()

## 13. Save Deployment Information

In [ ]:
# Save deployment information
deployment_info = {
    "baseline": {
        "endpoint_name": baseline_deployment["endpoint_name"],
        "model_path": baseline_s3_path,
        "inference_time": baseline_result["inference_time"],
        "model_size": baseline_size,
        "monthly_cost": baseline_cost["total_cost"]
    }
}

if quantized_deployment:
    deployment_info["quantized"] = {
        "endpoint_name": quantized_deployment["endpoint_name"],
        "model_path": quantized_s3_path,
        "inference_time": quantized_result["inference_time"],
        "model_size": quantized_size,
        "monthly_cost": quantized_cost["total_cost"],
        "speedup": speedup["Quantized"]
    }

if pruned_deployment:
    deployment_info["pruned"] = {
        "endpoint_name": pruned_deployment["endpoint_name"],
        "model_path": pruned_s3_path,
        "inference_time": pruned_result["inference_time"],
        "model_size": pruned_size,
        "monthly_cost": pruned_cost["total_cost"],
        "speedup": speedup["Pruned"]
    }

with open('deployment_info.json', 'w') as f:
    json.dump(deployment_info, f, indent=2)

print("Deployment information saved to deployment_info.json")

## 14. Next Steps

In this notebook, we've deployed our models to SageMaker and compared their inference performance and cost. In the next notebook, we'll clean up the resources we've created.